In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql.functions import *
from pyspark.sql.window import Window 

In [0]:
# if the csv data is saved as a file in a volume we will use this method to read the csv file into a dataframe
df = spark.read.format('csv').option('inferSchema',True).option("header",True).load("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/BigMart Sales.csv")

# Windows Functions

In [0]:
# row_number()
df.withColumn("Row_Number", row_number().over(Window.orderBy("Item_Identifier"))).limit(10).display()

In [0]:
# rank() and Dense_rank()
df.withColumn("Rank", rank().over(Window.orderBy("Item_Identifier")))\
    .withColumn("Dense_Rank", dense_rank().over(Window.orderBy("Item_Identifier"))).limit(10).display()

##### Cummulative sum

In [0]:
# Cummulative sum
df.withColumn("Cum_Sum", sum("Item_Outlet_Sales").over(Window.orderBy("Item_Type"))).limit(10).display()

In [0]:
#Cummulative sum
df.withColumn("Cum_Sum", sum("Item_Outlet_Sales").over(Window.orderBy("Item_Type").rowsBetween(Window.unboundedPreceding, Window.currentRow))).limit(10).display()

In [0]:
# Cummulative sum
df.withColumn("Cum_Sum", sum("Item_Outlet_Sales").over(Window.orderBy("Item_Type").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).limit(10).display()

# User Defined Functions (UDFs)

In [0]:
# User Defined Functions (UDFs)
def square(x):
    if x is None:
        return None
    return x * x


In [0]:
udf_square = udf(square)

In [0]:
df.withColumn("square", udf_square("Item_Outlet_Sales")).limit(10).display()

# Data Writing

### Modes of writing a Dataframe 
1. Append --> Add data to existing files
2. Overwrite --> Delete existing data and write fresh
3. Error -->Fail if path exists
4. Ignore --> Skip write if path exists

In [0]:
# Writing df into csv format using append
df.write.format("csv").mode("append").save("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/transformed/BigMart_Transformed.csv")

In [0]:
# Writing df into csv format using overwrite
df.write.format("csv").mode("overwrite").save("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/transformed/BigMart_Transformed.csv")

In [0]:
# Writing df into csv format using error
df.write.format("csv").mode("error").save("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/transformed/BigMart_Transformed.csv")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7037033852231760>, line 2
      1 # Writing df into csv format using error
----> 2 df.write.format("csv").mode("error").save("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/transformed/BigMart_Transformed.csv")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1556, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1554     req.user_co

In [0]:
# Writing df into csv format using ignore
df.write.format("csv").mode("ignore").save("/Volumes/workspace/pyspark_practice/pyspark_practice_volume/transformed/BigMart_Transformed.csv")

# File Formats

## 1. Columnar File Formats

#### Parquet
- Column-based storage
- Highly compressed
- Predicate pushdown
- Very fast for analytics

`df.write.parquet("/path/parquet")`


#### ORC

- Columnar (similar to Parquet)
- Better with Hive workloads
- Strong compression

`df.write.orc("/path/orc")`

## 2. Transactional File Formats

#### Delta Lake

- Built on Parquet
- ACID transactions
- Time travel
- Schema enforcement
- Supports MERGE / UPDATE / DELETE

`df.write.format("delta").save("/path/delta")`

# 3. Row-Based File Formats

#### CSV

- Human readable
- No schema
- Slow for analytics

`df.write.option("header","true").csv("/path/csv")`

#### JSON

- Semi-structured
- Nested support
- Large size

`df.write.json("/path/json")`

#### Text

- Raw text data
- No structure

`df.write.text("/path/text")`

# Comparison Table

In [0]:
%sql
> - Format	  Type	   ACID	   Compression	 Best Use
> - CSV	      Row	     ❌	   ❌	          Sharing
> - JSON	    Row    	 ❌	   ❌	          Logs
> - Parquet	  Column	 ❌	   ✅	          Analytics
> - ORC	      Column	 ❌	   ✅	          Hive
> - Delta	    Column	 ✅	   ✅	          Lakehouse
> - Avro	    Row	     ❌	   ✅	          Streaming
> - Hudi	    Column	 ✅	   ✅	          CDC
> - Iceberg	  Column	 ✅	   ✅	          Lakehouse